# Adaptive and Darcy-coupled transient transport

The stationary operator is conservative: $-\nabla\cdot(K\nabla u)+\nabla\cdot(\beta u)+cu=f$.
Its Robin multiplier is $(-K\nabla u+\beta u/2)\cdot n$. Strong Dirichlet data on essential faces and prescribed natural wall fluxes permit the face jump indicator of Harder, Paredes and Valentin (2015), DOI [10.1137/130938499](https://doi.org/10.1137/130938499).

This notebook executes a small adaptive example and reads the separate five-level numerical campaigns. It does not claim equality with historical table values or a discrete maximum principle.


The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/736a740e698b96bd1670be10c1769dbde0860fbfb17262460dc99a6cf6c1f28f/26_adaptive_transient_transport-companion.zip"
COMPANION_SHA256 = "736a740e698b96bd1670be10c1769dbde0860fbfb17262460dc99a6cf6c1f28f"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("transport/26_adaptive_transient_transport.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from pymhm import TriangleMesh, SkeletonSpace, assemble
from pymhm.adaptivity.transport import (
    TransportBounds, estimate_transport_faces, refine_skeleton_faces,
)
from examples.formulations.scalar import (
    ScalarDiscretization, strong_diffusion_equations,
    strong_diffusion_problem, recover_strong_scalar,
)

mesh = TriangleMesh.unit_square(2)
bounds = TransportBounds(1, 0, 0)
skeleton = SkeletonSpace(mesh)
natural = {int(f): 0.0 for f in mesh.boundary_faces if abs(mesh.normals[f, 1]) == 1}

def exact(points):
    return np.sin(np.pi * points[:, 0])

solutions, indicators = [], []
for iteration in range(3):
    data = ScalarDiscretization(mesh, skeleton, degree=2, refinement=4)
    problem = strong_diffusion_problem(
        data, source=lambda points: np.pi**2 * exact(points), diffusive_flux=natural,
    )
    system = assemble(problem)
    solution = recover_strong_scalar(data, system, system.solve(), natural_faces=tuple(natural))
    indicator = estimate_transport_faces(solution, bounds)
    solutions.append(solution)
    indicators.append(indicator)
    if indicator.total == 0:
        break
    if iteration < 2:
        skeleton = refine_skeleton_faces(skeleton, indicator.mark())

[(s.skeleton.size, eta.total, s.l2_error(exact, 7))
 for s, eta in zip(solutions, indicators)]


In [ ]:
from inspect import getsource
print(getsource(strong_diffusion_equations))
print(getsource(strong_diffusion_problem))


The mixed-wall campaign uses the Figure 12 physical data: $\epsilon=0.1$, $a=1$, $\sigma=0$, source one, strong zero Dirichlet data at $x=0,1$ and zero diffusive wall flux at $y=0,1$. Its local P1 Galerkin and P0 skeletal spaces match the published family. Sixteen initial crisscross macrotriangles and the local subdivision $r=16$ are specified choices; complete historical local connectivity is unavailable. Eight face-adaptive states, eight macro refinements and same-final-trace local controls are archived. The broken H1 error remains sensitive to local resolution.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    mixed = json.loads((root / "examples/results/transport/mixed-campaign.json").read_text())
    [(r["free_trace_dofs"], r["indicator"], r["quadrature"]["12"])
     for r in mixed["adaptive"]]

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    for name in ("mixed-convergence", "mixed-fields", "mixed-profiles"):
        display(Image(filename=str(root / f"docs/figures/transport/{name}.png")))

## Figure 12 and finite local resolution

The Figure 12 caption prints $K=0.1I$. Its absolute errors are compared without rescaling to the declared P1/Galerkin/P0 mixed-wall discretization. A separate control uses $K=I$, the regime explicitly published in Figure 7. Agreement of this different regime with graphical intervals does not identify the historical inputs or the cause of their incompatibility.

The comparison counts only free interior multipliers. Retained local constants are algebraic coordinates, recorded separately. The complete saddle systems for both diffusion regimes were also assembled independently in DOLFINx/UFL.


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    comparison = json.loads((root / "examples/results/transport/mixed-published-comparison.json").read_text())
    mesh_points = comparison["coefficient_mesh"]
    {
        "coefficient_regime": comparison["comparison_epsilon"],
        "caption_regime": comparison["printed_epsilon"],
        "compared_errors": len(mesh_points),
        "inside_graphical_intervals": sum(row["inside_graphical_interval"] for row in mesh_points),
        "norm_convention": comparison["norm_convention"],
    }


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    for name in ("mixed-published-comparison", "mixed-face-resolution", "mixed-local-resolution"):
        display(Image(filename=str(root / f"docs/figures/transport/{name}.png")))


Uniform face subdivision and maximum-indicator marking are distinct experiments. In the latter, $\theta=3/4$ selects individual segments while the macrogeometry remains fixed. Local P1 gradients belong to fine-cell DG0, so their distance from the exact gradient has a lower bound independent of skeletal enrichment. The two-quadrature values below are numerical integrations, not certified enclosures.

The fixed 1408-multiplier controls preserve all skeletal breakpoints at local subdivisions 16, 32, 64 and 128. Both error norms are reported; no monotonicity is imposed on the L2 error. The uniform family varies local resolution through 256 independently of its five skeletal subdivisions; every displayed point includes both integration orders. At subdivision 256, seven of ten errors lie inside the graphical intervals. Its final gradient error remains 59.28% above the published marker despite a DG0 lower bound below it. This separate epsilon=1 control does not reproduce the complete space curve or establish its historical inputs.


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    {
        "uniform_endpoint_errors_and_marker_ratios": {
            refinement: [(row["norm"], row["computed"], row["ratio_to_marker"])
                         for row in rows if row["free_trace_dofs"] == 5888]
            for refinement, rows in comparison["uniform_face_controls"].items()
        },
        "uniform_graphical_matches": {
            refinement: sum(row["inside_graphical_interval"] for row in rows)
            for refinement, rows in comparison["uniform_face_controls"].items()
        },
        "gradient_lower_bounds": comparison["gradient_projection_lower_bounds"],
        "fixed_adaptive_errors": [
            (row["local_refinement"], row["quadrature"]["12"])
            for row in comparison["fixed_adaptive_local_controls"]
        ],
    }


The successive field differences are measured on the actual nested triangles, independently of the difference between scalar error norms. At 5888 fixed multipliers, the r128→256 gradient increment is 2.2553158e−4, or 119.07% of the finest exact-gradient error. Local approximation is still material at this endpoint. The exact P1 mass and quadrature orders 8/12 agree; these are increments, not certified errors relative to the limiting local solution.


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    increments = json.loads((root / "examples/results/transport/mixed-local-field-increments.json").read_text())
    [(row["coarse_refinement"], row["fine_refinement"], row["segments"], row["norms"]["exact"])
     for row in increments["records"]]


The following full-Dirichlet control is separate: $\epsilon=0.02$, P3/SUPG locals and exact values on the entire boundary. Its data differ from the article wall conditions and from the P1/Galerkin mixed-wall study above. The intermediate nonmonotone errors remain in the record.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    record = json.loads((root / "examples/results/transport/campaign.json").read_text())
    record["adaptive"]


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    for name in ("refinement", "adaptive-0", "adaptive-4"):
        display(Image(filename=str(root / f"docs/figures/transport/{name}.png")))


## Transient Darcy coupling

The transport law is $\rho u_t-\nabla\cdot(D\nabla u)+\nabla\cdot(qu)+cu=f$, with RT0 Darcy velocity and the hydrodynamic dispersion law from section 5.4. Backward Euler includes the complete time residual in SUPG and reuses local/global factors for equal time increments.

The five-level manufactured verification uses two horizontal permeability layers on $[0,3]\times[0,1]$, $p=3-x$, $q=(K(y),0)$ and $u=e^{-t}[1+x(3-x)/9]$. This is an original exact-data check of the coupling, not a reconstruction of the paper's random permeability realization. The archived record separates physical error, Darcy flux error, operator reuse, and the discrete weak balance residual.


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    record["transient"]


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    display(Image(filename=str(root / "docs/figures/transport/darcy-transient.png")))


Acquire the mixed-wall study with `python -m examples.transport_mixed_campaign`, then replay with `python -m examples.plot_transport_mixed`. The separate P3 and transient controls use `python -m examples.transport_campaign --collect`. Source notebooks remain unexecuted; executable outputs belong in the notebook build directory.